# MADS — Threat Detection Neural Network Classifier

This notebook builds a binary neural network classifier to detect malicious activity in a cybersecurity event log dataset.

## Objective
Classify each event as **malicious** or **benign** based on five numerical features (`feature_1` to `feature_5`) extracted from machine activity logs.

## Dataset
- **Source:** `mads_synthetic_dataset.csv`
- **Rows:** ~142 events across 5 machines
- **Features:** `feature_1`, `feature_2`, `feature_3`, `feature_4`, `feature_5`
- **Label:** `malicious` / `benign`
- **Event types include:** data exfiltration, lateral movement, privilege escalation, C2 connections, and normal activity

## Approach
| Step | Description |
|---|---|
| **Data Preparation** | Feature extraction, label encoding, 70/30 stratified train/test split, StandardScaler normalisation |
| **Model** | 3-layer fully-connected neural network (5 → 32 → 16 → 1) with ReLU activations and Dropout regularisation |
| **Training** | Binary cross-entropy loss, Adam optimiser, 20 epochs with tqdm progress tracking |
| **Evaluation** | Accuracy, precision, recall, and F1-score on the held-out test set |
| **Cross-Validation** | 5-fold stratified CV to produce a robust performance estimate (mean ± std accuracy) |

In [5]:
!pip install torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.1 MB/s  0:04:16 eta 0:00:010:00:06m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 3.5 MB/s  0:02:33 eta 0:00:010:00:03m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 1.6 MB/s  0:00:51 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 2.4 MB/s  0:01:00 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 2.2 MB/s  0:00:27 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 1.3 MB/s  0:01:51 eta 0:00:010:00:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 1.7 MB/s  0:00:03a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.2 MB/s  0:03:01 eta 0:00:010:00:06
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 3.8 MB/s  0:00:02a 0:00:0136m0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 1.9 MB/s  0:00:41 eta 0:00:010:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("../datasets/mads_synthetic_dataset.csv")

In [3]:
df.head()

,event_id,timestamp,machine,event_type,feature_1,feature_2,feature_3,feature_4,feature_5,ip_address,user_id,process,label
0,EVT-00001,2025-04-13 12:00:00.000000,machine1,data_exfiltration,0.7266,0.7384,-0.1083,0.7594,0.9460,192.168.1.20,U1013,python.exe,malicious
1,EVT-00002,2025-04-13 12:00:11.459098,machine3,normal_activity,0.4288,0.5284,0.4184,0.4572,0.8116,192.168.1.13,U1006,whoami.exe,benign
2,EVT-00003,2025-04-13 12:01:32.010817,machine4,suspicious_file_download,0.5295,0.7131,0.5481,0.9290,-0.0716,192.168.1.30,U1015,svchost.exe,malicious
3,EVT-00004,2025-04-13 12:02:30.953552,machine1,lateral_movement,0.4490,-0.0086,0.9424,0.4923,0.5986,192.168.1.15,U1013,rundll32.exe,malicious
4,EVT-00005,2025-04-13 12:02:48.231682,machine1,c2_connection,0.5861,0.5747,0.3330,0.4468,0.9615,192.168.1.23,U1008,netstat.exe,malicious


In [4]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from tqdm.notebook import tqdm

## Data Preparation

In [5]:
# Select features and encode label
feature_cols = ['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5']

X = df[feature_cols].values.astype(np.float32)
y = (df['label'] == 'malicious').astype(np.float32).values  # 1 = malicious, 0 = benign

# Train / test split (70/30, stratified to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Fit scaler on training set only to prevent data leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test  = scaler.transform(X_test).astype(np.float32)

print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")
print(f"Class balance (train) — benign: {(y_train == 0).sum()}, malicious: {(y_train == 1).sum()}")

Train: 4200 samples | Test: 1800 samples
Class balance (train) — benign: 1455, malicious: 2745


In [7]:

# Class distribution in train and test sets
for split_name, y_split in [("y_train", y_train), ("y_test", y_test)]:
    benign    = (y_split == 0).sum()
    malicious = (y_split == 1).sum()
    total     = len(y_split)
    print(f"{split_name}  —  benign: {benign} ({benign/total:.1%})  |  malicious: {malicious} ({malicious/total:.1%})  |  total: {total}")


y_train  —  benign: 1455 (34.6%)  |  malicious: 2745 (65.4%)  |  total: 4200
y_test  —  benign: 624 (34.7%)  |  malicious: 1176 (65.3%)  |  total: 1800


## PyTorch Dataset & DataLoaders

In [8]:
class SecurityDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_dataset = SecurityDataset(X_train, y_train)
test_dataset  = SecurityDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)} | Test batches: {len(test_loader)}")

Train batches: 132 | Test batches: 57


## Neural Network Model

In [9]:
class ThreatClassifier(nn.Module):
    def __init__(self, input_dim: int = 5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(16, 1),   # raw logit — BCEWithLogitsLoss handles sigmoid
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = ThreatClassifier(input_dim=5).to(device)

print(model)
print(f"\nRunning on: {device}")

ThreatClassifier(
  (net): Sequential(
    (0): Linear(in_features=5, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=16, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=16, out_features=1, bias=True)
  )
)

Running on: cpu


## Training

In [10]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20
train_losses = []

epoch_bar = tqdm(range(1, epochs + 1), desc="Training", unit="epoch")

for epoch in epoch_bar:
    model.train()
    running_loss = 0.0

    batch_bar = tqdm(train_loader, desc=f"  Epoch {epoch:3d}/{epochs}", leave=False, unit="batch")

    for X_batch, y_batch in batch_bar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(X_batch)
        batch_bar.set_postfix(batch_loss=f"{loss.item():.4f}")

    avg_loss = running_loss / len(train_dataset)
    train_losses.append(avg_loss)
    epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}")
    tqdm.write(f"Epoch {epoch:3d}/{epochs}  |  Avg Loss: {avg_loss:.4f}")

print("\nTraining complete.")
print("\n--- Loss History ---")
print(f"{'Epoch':>6}  {'Avg Loss':>10}")
print("-" * 20)
for i, loss_val in enumerate(train_losses, 1):
    print(f"{i:>6}  {loss_val:>10.4f}")

Training:   0%|          | 0/20 [00:00<?, ?epoch/s]

  Epoch   1/20:   0%|          | 0/132 [00:00<?, ?batch/s]

Epoch   1/20  |  Avg Loss: 0.5736


  Epoch   2/20:   0%|          | 0/132 [00:00<?, ?batch/s]

Epoch   2/20  |  Avg Loss: 0.4602


...

Epoch  18/20  |  Avg Loss: 0.4370


  Epoch  19/20:   0%|          | 0/132 [00:00<?, ?batch/s]

Epoch  19/20  |  Avg Loss: 0.4381


  Epoch  20/20:   0%|          | 0/132 [00:00<?, ?batch/s]

Epoch  20/20  |  Avg Loss: 0.4344

Training complete.

--- Loss History ---
 Epoch    Avg Loss
--------------------
     1      0.5736
     2      0.4602
     3      0.4522
     4      0.4481
     5      0.4468
     6      0.4440
     7      0.4419
     8      0.4422
     9      0.4407
    10      0.4397
    11      0.4356
    12      0.4356
    13      0.4387
    14      0.4356
    15      0.4321
    16      0.4389
    17      0.4352
    18      0.4370
    19      0.4381
    20      0.4344


## Evaluation

In [11]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch.to(device))
        preds  = (torch.sigmoid(logits) >= 0.5).cpu().int().tolist()
        all_preds.extend(preds)
        all_labels.extend(y_batch.int().tolist())

print(f"Test Accuracy: {accuracy_score(all_labels, all_preds):.4f}\n")
print(classification_report(all_labels, all_preds, target_names=['benign', 'malicious']))

Test Accuracy: 0.7978

              precision    recall  f1-score   support

      benign       0.71      0.69      0.70       624
   malicious       0.84      0.85      0.85      1176

    accuracy                           0.80      1800
   macro avg       0.78      0.77      0.78      1800
weighted avg       0.80      0.80      0.80      1800



## Cross-Validation

In [12]:
from sklearn.model_selection import StratifiedKFold

# Re-use raw (unscaled) X and y — scaling is applied fresh inside each fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracies = []
cv_losses     = []

print(f"Running {skf.n_splits}-fold Stratified Cross-Validation...")
print(f"{'Fold':>5}  {'Best Loss':>10}  {'Accuracy':>10}")
print("-" * 32)

Running 5-fold Stratified Cross-Validation...
 Fold   Best Loss    Accuracy
--------------------------------


In [13]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):

    # ── 1. Split ──────────────────────────────────────────────
    X_tr,  X_val  = X[train_idx], X[val_idx]
    y_tr,  y_val  = y[train_idx], y[val_idx]

    # ── 2. Scale (fit on train fold only) ─────────────────────
    fold_scaler = StandardScaler()
    X_tr  = fold_scaler.fit_transform(X_tr).astype(np.float32)
    X_val = fold_scaler.transform(X_val).astype(np.float32)

    # ── 3. DataLoaders ────────────────────────────────────────
    fold_train_loader = DataLoader(SecurityDataset(X_tr,  y_tr),  batch_size=32, shuffle=True)
    fold_val_loader   = DataLoader(SecurityDataset(X_val, y_val), batch_size=32, shuffle=False)

    # ── 4. Fresh model & optimizer per fold ───────────────────
    fold_model     = ThreatClassifier(input_dim=5).to(device)
    fold_optimizer = torch.optim.Adam(fold_model.parameters(), lr=1e-3)
    fold_criterion = nn.BCEWithLogitsLoss()

    # ── 5. Train ──────────────────────────────────────────────
    fold_losses = []
    epoch_bar   = tqdm(range(1, epochs + 1), desc=f"Fold {fold}", unit="epoch", leave=False)

    for epoch in epoch_bar:
        fold_model.train()
        running_loss = 0.0

        for X_batch, y_batch in fold_train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            fold_optimizer.zero_grad()
            loss = fold_criterion(fold_model(X_batch), y_batch)
            loss.backward()
            fold_optimizer.step()
            running_loss += loss.item() * len(X_batch)

        avg_loss = running_loss / len(X_tr)
        fold_losses.append(avg_loss)
        epoch_bar.set_postfix(avg_loss=f"{avg_loss:.4f}")

    # ── 6. Evaluate on validation fold ────────────────────────
    fold_model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for X_batch, y_batch in fold_val_loader:
            logits = fold_model(X_batch.to(device))
            preds.extend((torch.sigmoid(logits) >= 0.5).cpu().int().tolist())
            labels.extend(y_batch.int().tolist())

    acc       = accuracy_score(labels, preds)
    best_loss = min(fold_losses)
    cv_accuracies.append(acc)
    cv_losses.append(best_loss)

    print(f"{fold:>5}  {best_loss:>10.4f}  {acc:>10.4f}")

Fold 1:   0%|          | 0/20 [00:00<?, ?epoch/s]

    1      0.4246      0.7975


Fold 2:   0%|          | 0/20 [00:00<?, ?epoch/s]

    2      0.4285      0.8075


Fold 3:   0%|          | 0/20 [00:00<?, ?epoch/s]

    3      0.4220      0.7967


Fold 4:   0%|          | 0/20 [00:00<?, ?epoch/s]

    4      0.4271      0.8042


Fold 5:   0%|          | 0/20 [00:00<?, ?epoch/s]

    5      0.4225      0.7875


In [14]:
# ── Summary ───────────────────────────────────────────────────
print("-" * 32)
print(f"{'Mean':>5}  {np.mean(cv_losses):>10.4f}  {np.mean(cv_accuracies):>10.4f}")
print(f"{'Std':>5}  {np.std(cv_losses):>10.4f}  {np.std(cv_accuracies):>10.4f}")
print(f"\nCV Accuracy: {np.mean(cv_accuracies):.4f} ± {np.std(cv_accuracies):.4f}")

--------------------------------
 Mean      0.4249      0.7987
  Std      0.0025      0.0069

CV Accuracy: 0.7987 ± 0.0069
